In [ ]:
print("kernel alive")

In [ ]:
from experiments.robot.libero.gen_config import GenerateConfig
from experiments.robot.openvla_utils import get_action_head, get_processor, get_proprio_projector, get_vla
from prismatic.vla.constants import NUM_ACTIONS_CHUNK, PROPRIO_DIM


# Instantiate config (see class GenerateConfig in experiments/robot/libero/run_libero_eval.py for definitions)
cfg = GenerateConfig(
    pretrained_checkpoint = "openvla_7b_oft_finetuned",
    use_l1_regression = True,
    use_diffusion = False,
    use_film = False,
    num_images_in_input = 2,
    use_proprio = True,
    load_in_8bit = False,
    load_in_4bit = True,
    center_crop = True,
    num_open_loop_steps = NUM_ACTIONS_CHUNK,
    unnorm_key = "libero_spatial_no_noops",
)

# Load OpenVLA-OFT policy and inputs processor
vla  = get_vla(cfg)
processor = get_processor(cfg)

# Load MLP action head to generate continuous actions (via L1 regression)
action_head = get_action_head(cfg, llm_dim=vla.llm_dim)

# Load proprio projector to map proprio to language embedding space
proprio_projector = get_proprio_projector(cfg, llm_dim=vla.llm_dim, proprio_dim=PROPRIO_DIM)

In [ ]:
import pickle
from experiments.robot.openvla_utils import get_vla_action

with open("experiments/robot/libero/sample_libero_spatial_observation.pkl", "rb") as file:
    observation = pickle.load(file)

# Generate robot action chunk (sequence of future actions)
actions = get_vla_action(cfg, vla, processor, observation, observation["task_description"], action_head, proprio_projector)
print("Generated action chunk:")
for act in actions:
    print(act)

### Batch mask attention Genneration

In [ ]:
import torch

IGNORE_INDEX = -100
ACTION_TOKEN_ID = 29871

In [ ]:
def make_prompts(task_labels):
    return [
        f"In: What action should the robot take to {task.lower()}?\nOut:"
        for task in task_labels
    ]

def make_text_batch(task_labels, processor, device):
    prompts = make_prompts(task_labels)

    encoded = [
        processor.tokenizer(prompt, return_tensors="pt")
        for prompt in prompts
    ]

    input_ids = torch.cat([x["input_ids"] for x in encoded], dim=0).to(device)
    attention_mask = torch.cat([x["attention_mask"] for x in encoded], dim=0).to(device)

    return input_ids.long(), attention_mask.long(), prompts

In [ ]:
from experiments.robot.openvla_utils import prepare_images_for_vla

def make_pixel_batch(obs_batch, prompts, cfg, processor, device, dtype):
    pixel_values = []

    for obs, prompt in zip(obs_batch, prompts):
        images = [obs["full_image"]]

        if cfg.num_images_in_input > 1:
            images += [v for k, v in obs.items() if "wrist" in k]

        images = prepare_images_for_vla(images, cfg)

        inputs = processor(prompt, images[0]).to(device, dtype=dtype)

        if len(images) > 1:
            wrist_pixels = [
                processor(prompt, img).to(device, dtype=dtype)["pixel_values"]
                for img in images[1:]
            ]
            inputs["pixel_values"] = torch.cat(
                [inputs["pixel_values"]] + wrist_pixels,
                dim=1,
            )

        pixel_values.append(inputs["pixel_values"])

    return torch.cat(pixel_values, dim=0)

In [ ]:
def make_action_inputs(input_ids, attention_mask, vla):
    if not torch.all(input_ids[:, -1] == ACTION_TOKEN_ID):
        extra_ids = torch.full(
            (input_ids.shape[0], 1),
            ACTION_TOKEN_ID,
            device=input_ids.device,
            dtype=input_ids.dtype,
        )
        extra_mask = torch.ones(
            (attention_mask.shape[0], 1),
            device=attention_mask.device,
            dtype=attention_mask.dtype,
        )
        input_ids = torch.cat([input_ids, extra_ids], dim=1)
        attention_mask = torch.cat([attention_mask, extra_mask], dim=1)

    return vla._prepare_input_for_action_prediction(input_ids, attention_mask)

In [ ]:
@torch.no_grad()
def make_vla_batch(batch, cfg, vla, processor, device=None, dtype=torch.bfloat16):
    device = device or next(vla.parameters()).device

    input_ids, attention_mask, prompts = make_text_batch(
        batch["task_labels"], processor, device
    )

    pixel_values = make_pixel_batch(
        batch["obs_batch"], prompts, cfg, processor, device, dtype
    )

    input_ids, attention_mask = make_action_inputs(
        input_ids, attention_mask, vla
    )

    labels = torch.full_like(input_ids, IGNORE_INDEX)
    labels = vla._prepare_labels_for_action_prediction(labels, input_ids)

    all_actions_mask = vla._process_action_masks(labels)
    input_embeddings = vla.get_input_embeddings()(input_ids)

    language_embeddings = input_embeddings[~all_actions_mask].reshape(
        input_embeddings.shape[0],
        -1,
        input_embeddings.shape[-1],
    )

    projected_patch_embeddings = vla._process_vision_features(
        pixel_values,
        language_embeddings,
        False,
    )

    return {
        "projected_patch_embeddings": projected_patch_embeddings,
        "input_embeddings": input_embeddings,
        "all_actions_mask": all_actions_mask,
        "attention_mask": attention_mask,
        "labels": labels,
        "input_ids": input_ids,
        "pixel_values": pixel_values,
    }

### TSO loading functions

In [ ]:
import re
import h5py
import torch
from pathlib import Path
from torch.utils.data import Dataset, DataLoader


def task_name_from_path(path):
    stem = Path(path).stem
    stem = re.sub(r"_demo$", "", stem)
    stem = re.sub(r"^(?:[A-Z0-9]+_)+(?=[a-z])", "", stem)
    return stem.replace("_", " ")


def demo_sort_key(demo_id):
    m = re.search(r"\d+$", demo_id)
    return int(m.group()) if m else demo_id


def load_tso_by_demo(tso_dir):
    tso_dir = Path(tso_dir)
    out = {}

    for p in sorted(tso_dir.glob("*.pt")):
        data = torch.load(p, map_location="cpu")

        demo_id = data.get("source_demo_name", p.stem)
        features = data["features"]
        frame_indices = data.get("frame_indices", None)

        if frame_indices is None:
            out[demo_id] = features
        else:
            out[demo_id] = {
                int(frame_idx): feat
                for frame_idx, feat in zip(frame_indices, features)
            }

    return out

def load_libero_frames(path):
    path = Path(path)
    task = task_name_from_path(path)
    samples = []

    with h5py.File(path, "r") as f:
        for demo_id in sorted(f["data"].keys(), key=demo_sort_key):
            obs = f["data"][demo_id]["obs"]
            agentview = obs["agentview_rgb"][()]
            wrist = obs["eye_in_hand_rgb"][()]

            for step in range(agentview.shape[0]):
                samples.append({
                    "obs": {
                        "full_image": agentview[step],
                        "wrist_image": wrist[step],
                    },
                    "task_label": task,
                    "demo_id": demo_id,
                    "step": step,
                })

    print(f"loaded {len(samples)} frames | task: {task}")
    return samples


class FrameDataset(Dataset):
    def __init__(self, samples, tso_by_demo=None):
        self.samples = samples
        self.tso_by_demo = tso_by_demo

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, i):
        x = self.samples[i]

        out = dict(x)

        if self.tso_by_demo is not None:
            tso_demo = self.tso_by_demo[x["demo_id"]]

            if isinstance(tso_demo, dict):
                out["tso"] = tso_demo[x["step"]]
            else:
                out["tso"] = tso_demo[x["step"]]

        return out


def collate_frames(batch):
    out = {
        "obs_batch": [x["obs"] for x in batch],
        "task_labels": [x["task_label"] for x in batch],
        "demo_ids": [x["demo_id"] for x in batch],
        "steps": [x["step"] for x in batch],
    }

    if "tso" in batch[0]:
        out["tso"] = torch.stack([x["tso"] for x in batch])

    return out

def make_libero_dataloader(
    data_path,
    batch_size=16,
    shuffle=False,
    num_workers=0,
    tso_dir=None,
):
    samples = load_libero_frames(data_path)
    tso_by_demo = load_tso_by_demo(tso_dir) if tso_dir is not None else None

    dataset = FrameDataset(samples, tso_by_demo=tso_by_demo)

    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=num_workers,
        collate_fn=collate_frames,
    )

In [ ]:
loader = make_libero_dataloader(
    data_path="libero_data/libero_spatial/pick_up_the_black_bowl_on_the_wooden_cabinet_and_place_it_on_the_plate_demo.hdf5",
    tso_dir="tso_data/libero_spatial/pick_up_the_black_bowl_on_the_wooden_cabinet_and_place_it_on_the_plate",
    batch_size=8,
)

In [ ]:
batch = next(iter(loader))

clue = batch["tso"].to(
    device=next(vla.parameters()).device,
    dtype=torch.bfloat16,
)

### Projected Patch Embedding Generation

In [ ]:
import torch
from experiments.robot.openvla_utils import prepare_images_for_vla

IGNORE_INDEX = -100

@torch.no_grad()
def get_projected_patch_embeddings_batchwise(
    obs_batch,
    task_labels,
    cfg,
    vla,
    processor,
    device=None,
    dtype=torch.bfloat16,
):
    if device is None:
        device = next(vla.parameters()).device

    input_ids_list = []
    attention_mask_list = []
    pixel_values_list = []

    for obs, task_label in zip(obs_batch, task_labels):
        all_images = [obs["full_image"]]

        if cfg.num_images_in_input > 1:
            all_images.extend([obs[k] for k in obs.keys() if "wrist" in k])

        all_images = prepare_images_for_vla(all_images, cfg)
        primary_image = all_images.pop(0)

        prompt = f"In: What action should the robot take to {task_label.lower()}?\nOut:"

        inputs = processor(prompt, primary_image).to(device, dtype=dtype)

        if len(all_images) > 0:
            wrist_inputs = [
                processor(prompt, img).to(device, dtype=dtype)
                for img in all_images
            ]

            inputs["pixel_values"] = torch.cat(
                [inputs["pixel_values"]]
                + [x["pixel_values"] for x in wrist_inputs],
                dim=1,
            )

        input_ids_list.append(inputs["input_ids"].long())
        attention_mask_list.append(inputs["attention_mask"].long())
        pixel_values_list.append(inputs["pixel_values"])

    input_ids = torch.cat(input_ids_list, dim=0)
    attention_mask = torch.cat(attention_mask_list, dim=0)
    pixel_values = torch.cat(pixel_values_list, dim=0)

    if not torch.all(input_ids[:, -1] == 29871):
        extra_token = torch.full(
            (input_ids.shape[0], 1),
            29871,
            device=device,
            dtype=input_ids.dtype,
        )
        input_ids = torch.cat([input_ids, extra_token], dim=1)

        extra_mask = torch.ones(
            (attention_mask.shape[0], 1),
            device=device,
            dtype=attention_mask.dtype,
        )
        attention_mask = torch.cat([attention_mask, extra_mask], dim=1)

    labels = input_ids.clone()
    labels[:] = IGNORE_INDEX

    input_ids, attention_mask = vla._prepare_input_for_action_prediction(
        input_ids,
        attention_mask,
    )

    labels = vla._prepare_labels_for_action_prediction(labels, input_ids)
    all_actions_mask = vla._process_action_masks(labels)

    input_embeddings = vla.get_input_embeddings()(input_ids)

    language_embeddings = input_embeddings[~all_actions_mask].reshape(
        input_embeddings.shape[0],
        -1,
        input_embeddings.shape[2],
    )

    projected_patch_embeddings = vla._process_vision_features(
        pixel_values,
        language_embeddings,
        False,
    )

    return projected_patch_embeddings

In [ ]:
patch_tokens = get_projected_patch_embeddings_batchwise(
    obs_batch=batch["obs_batch"],
    task_labels=batch["task_labels"],
    cfg=cfg,
    vla=vla,
    processor=processor,
)

### FiLM

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class ReViPTSFiLM(nn.Module):
    def __init__(
        self,
        clue_dim=2048,
        token_dim=4096,
        proj_hidden=2048,
        film_hidden=2048,
        gamma_scale=0.1,
        beta_scale=0.1,
    ):
        super().__init__()

        # Proj(.): clue -> token space
        self.clue_proj = nn.Sequential(
            nn.Linear(clue_dim, proj_hidden),
            nn.SiLU(),
            nn.Linear(proj_hidden, token_dim),
        )

        # h(.): z -> [gamma, beta]
        self.film_head = nn.Sequential(
            nn.Linear(token_dim, film_hidden),
            nn.SiLU(),
            nn.Linear(film_hidden, 2 * token_dim),
        )

        # Learnable nonnegative alpha through softplus
        # Start very small so TS-FiLM is near identity at initialization
        self.alpha_raw = nn.Parameter(torch.tensor(-0.43))

        self.gamma_scale = gamma_scale
        self.beta_scale = beta_scale

        self.reset_parameters()

    def reset_parameters(self):
        nn.init.zeros_(self.film_head[-1].bias)
        nn.init.xavier_uniform_(self.film_head[-1].weight, gain=0.01)

    def forward(self, patch_tokens, clue, return_aux=False):
        """
        patch_tokens: [B, N, D]
        clue:         [B, clue_dim]
        """
        if patch_tokens.dim() != 3:
            raise ValueError(f"patch_tokens must be [B, N, D], got {patch_tokens.shape}")
        if clue.dim() != 2:
            raise ValueError(f"clue must be [B, clue_dim], got {clue.shape}")

        B, N, D = patch_tokens.shape
        if clue.shape[0] != B:
            raise ValueError("Batch size mismatch between patch_tokens and clue")

        if D != self.film_head[-1].out_features // 2:
            raise ValueError(
                f"Token dim mismatch: patch_tokens has D={D}, "
                f"but TS-FiLM expects {self.film_head[-1].out_features // 2}"
            )

        z = self.clue_proj(clue)                    # [B, D]
        gamma_beta = self.film_head(z)             # [B, 2D]
        gamma, beta = gamma_beta.chunk(2, dim=-1)  # [B, D], [B, D]

        # Keep FiLM modulation bounded
        gamma = self.gamma_scale * torch.tanh(gamma)
        beta = self.beta_scale * torch.tanh(beta)

        alpha = F.softplus(self.alpha_raw)         # scalar >= 0

        delta = gamma[:, None, :] * patch_tokens + beta[:, None, :]
        mod_patch_tokens = patch_tokens + alpha * delta

        if return_aux:
            return mod_patch_tokens, {
                "z": z,
                "gamma": gamma,
                "beta": beta,
                "alpha": alpha,
            }

        return mod_patch_tokens
    
import torch
import torch.nn.functional as F


def revip_action_loss(
    pred_actions: torch.Tensor,
    gt_actions: torch.Tensor,
    loss_type: str = "l1",
):
    """
    Loss on generated actions vs ground-truth actions.

    Args:
        pred_actions: [B, ...]
        gt_actions:   [B, ...] same shape as pred_actions
        loss_type:    "l1", "mse", or "smooth_l1"

    Returns:
        loss: scalar tensor
        stats: dict
    """
    if pred_actions.shape != gt_actions.shape:
        raise ValueError(
            f"Shape mismatch: pred_actions {pred_actions.shape} vs gt_actions {gt_actions.shape}"
        )

    if loss_type == "l1":
        loss = F.l1_loss(pred_actions, gt_actions)
    elif loss_type == "mse":
        loss = F.mse_loss(pred_actions, gt_actions)
    elif loss_type == "smooth_l1":
        loss = F.smooth_l1_loss(pred_actions, gt_actions)
    else:
        raise ValueError(f"Unsupported loss_type: {loss_type}")

    stats = {
        "loss": loss.detach(),
        "pred_mean_abs": pred_actions.abs().mean().detach(),
        "gt_mean_abs": gt_actions.abs().mean().detach(),
        "error_mean_abs": (pred_actions - gt_actions).abs().mean().detach(),
    }

    return loss, stats

In [ ]:
film = ReViPTSFiLM(
    clue_dim=clue.shape[-1],
    token_dim=patch_tokens.shape[-1],
).to(patch_tokens.device, dtype=patch_tokens.dtype)

mod_patch_tokens, aux = film(
    patch_tokens,
    clue,
    return_aux=True,
)

print(patch_tokens.shape)
print(clue.shape)
print(mod_patch_tokens.shape)
print(aux["alpha"])